# 14 — Integrated Statistical Comparison

Bring together the primary random-versus-grouped comparison, grouped-fold diagnostics, and a host-star-cluster bootstrap of paired metric differences.

**Reproducibility note:** This notebook follows Notebook 03's primary sample definition and Random Forest pipeline. It does not overwrite or modify any earlier notebook. Run cells from top to bottom in Kaggle or Jupyter.

In [1]:
import os, glob, warnings
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.metrics import (
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, precision_score, recall_score, f1_score, log_loss
)

# Match Notebook 03's primary population and modeling settings.
RADIUS_MIN, RADIUS_MAX = 1.0, 4.0
TEFF_MIN, TEFF_MAX = 4700, 6500
N_SPLITS = 5
THRESHOLD = 0.50
RANDOM_STATE = 42
N_ESTIMATORS = 400

# Set this manually if auto-detection finds zero or multiple candidate CSV files.
DATA_PATH = None  # e.g. "/kaggle/input/your-dataset/cumulative.csv"

def locate_koi_csv():
    if DATA_PATH is not None:
        if not os.path.exists(DATA_PATH):
            raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")
        return DATA_PATH
    candidates = sorted(
        p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True)
        if ("cumulative" in os.path.basename(p).lower()
            or "koi" in os.path.basename(p).lower())
    )
    if len(candidates) == 1:
        return candidates[0]
    if not candidates:
        raise FileNotFoundError(
            "No KOI cumulative CSV found under /kaggle/input. "
            "Attach the same catalog used in Notebook 03 and set DATA_PATH manually."
        )
    print("Possible KOI CSV files:")
    for p in candidates:
        print(" -", p)
    raise ValueError("Multiple possible KOI CSVs found. Set DATA_PATH explicitly.")

def load_main_sample():
    path = locate_koi_csv()
    print("Using catalog:", path)
    raw = pd.read_csv(path, comment="#", low_memory=False)
    required = {"kepid", "koi_disposition", "koi_prad", "koi_steff"}
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"Required columns missing: {sorted(missing)}")
    for col in ["kepid", "koi_prad", "koi_steff"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    catalog = raw[
        raw["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE", "CANDIDATE"])
        & raw["kepid"].notna()
        & raw["koi_prad"].between(RADIUS_MIN, RADIUS_MAX, inclusive="both")
        & raw["koi_steff"].between(TEFF_MIN, TEFF_MAX, inclusive="both")
    ].copy()
    catalog["group"] = catalog["kepid"].astype("int64")
    labeled = catalog[
        catalog["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])
    ].copy()
    labeled["label"] = (labeled["koi_disposition"] == "CONFIRMED").astype(int)
    candidate_features = [
        "koi_period", "koi_time0bk", "koi_impact", "koi_duration",
        "koi_depth", "koi_prad", "koi_teq", "koi_insol",
        "koi_steff", "koi_slogg", "koi_srad", "koi_kepmag"
    ]
    features = [c for c in candidate_features if c in labeled.columns]
    if len(features) < 3:
        raise ValueError(f"Fewer than 3 Notebook 03 features available: {features}")
    X = labeled[features].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    drop = [c for c in X if X[c].notna().sum() == 0 or X[c].nunique(dropna=True) <= 1]
    if drop:
        print("Dropping all-missing/constant features:", drop)
        X = X.drop(columns=drop)
    y = labeled["label"].to_numpy(dtype=int)
    groups = labeled["group"].to_numpy()
    if len(np.unique(y)) != 2:
        raise ValueError("Both CONFIRMED and FALSE POSITIVE labels are required.")
    if labeled.groupby("label")["group"].nunique().min() < N_SPLITS:
        raise ValueError("Too few distinct host stars in at least one class for 5 folds.")
    print(f"Filtered catalog rows: {len(catalog):,}")
    print(f"Resolved labeled rows: {len(labeled):,}")
    print(f"Unique resolved host stars: {len(np.unique(groups)):,}")
    print("Class counts (0=false positive, 1=confirmed):")
    print(pd.Series(y).value_counts().sort_index())
    print("Features:", list(X.columns))
    return path, raw, catalog, labeled, X, y, groups

def build_model(seed=RANDOM_STATE):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("classifier", RandomForestClassifier(
            n_estimators=N_ESTIMATORS,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=seed,
            n_jobs=-1
        ))
    ])

def make_oof(X, y, groups, grouped=True, seed=RANDOM_STATE, n_splits=N_SPLITS):
    splitter = (
        StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        if grouped else
        StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    )
    probs = np.full(len(y), np.nan, dtype=float)
    folds = np.full(len(y), -1, dtype=int)
    split_iter = splitter.split(X, y, groups) if grouped else splitter.split(X, y)
    for fold, (tr, te) in enumerate(split_iter):
        if np.unique(y[tr]).size != 2 or np.unique(y[te]).size != 2:
            raise ValueError(
                f"Fold {fold+1} has a single class. Inspect group distribution; "
                "do not silently report fold-level class metrics."
            )
        if grouped and set(groups[tr]).intersection(set(groups[te])):
            raise RuntimeError(f"Host-star overlap in fold {fold+1}.")
        model = build_model(seed=seed)
        model.fit(X.iloc[tr], y[tr])
        probs[te] = model.predict_proba(X.iloc[te])[:, 1]
        folds[te] = fold
    if np.isnan(probs).any() or (folds < 0).any():
        raise RuntimeError("Every row must receive exactly one out-of-fold prediction.")
    return probs, folds

def metric_dict(y, p, threshold=THRESHOLD):
    pred = (p >= threshold).astype(int)
    return {
        "ROC_AUC": roc_auc_score(y, p),
        "average_precision": average_precision_score(y, p),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "Brier_score_lower_is_better": brier_score_loss(y, p),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "F1": f1_score(y, pred, zero_division=0),
        "threshold": threshold
    }

def save_csv(df, filename):
    os.makedirs("/kaggle/working", exist_ok=True)
    path = os.path.join("/kaggle/working", filename)
    df.to_csv(path, index=False)
    print("Saved:", path)

## Aim
Provide an integrated statistical comparison of random-row and host-star-grouped validation on the Notebook 03 main sample. The primary comparison uses one paired set of out-of-fold predictions (seed 42). A host-cluster bootstrap resamples whole host stars, keeping all rows from each sampled host together.

In [2]:
path, raw, catalog, labeled, X, y, groups = load_main_sample()
random_p, random_fold = make_oof(X, y, groups, grouped=False, seed=RANDOM_STATE)
grouped_p, grouped_fold = make_oof(X, y, groups, grouped=True, seed=RANDOM_STATE)

primary = pd.DataFrame([
    {"validation": "Random row-level CV", **metric_dict(y, random_p)},
    {"validation": "Host-star grouped CV", **metric_dict(y, grouped_p)}
])
display(primary.round(5))
save_csv(primary, "14_integrated_primary_metrics.csv")

# Fold diagnostics for grouped validation.
fold_rows = []
for fold in range(N_SPLITS):
    mask = grouped_fold == fold
    fold_rows.append({
        "fold": fold + 1,
        "n_test_rows": int(mask.sum()),
        "n_test_host_stars": int(pd.Series(groups[mask]).nunique()),
        "n_confirmed": int(y[mask].sum()),
        "n_false_positive": int((y[mask] == 0).sum()),
        "ROC_AUC": roc_auc_score(y[mask], grouped_p[mask])
            if np.unique(y[mask]).size == 2 else np.nan,
        "average_precision": average_precision_score(y[mask], grouped_p[mask])
            if np.unique(y[mask]).size == 2 else np.nan,
        "balanced_accuracy": balanced_accuracy_score(y[mask], grouped_p[mask] >= THRESHOLD)
            if np.unique(y[mask]).size == 2 else np.nan
    })
fold_diagnostics = pd.DataFrame(fold_rows)
display(fold_diagnostics.round(4))
save_csv(fold_diagnostics, "14_grouped_fold_diagnostics.csv")

Using catalog: /kaggle/input/datasets/sairakeshnagaram/koi-dataset/cumulative_2026.10.20_19.25.07.csv
Filtered catalog rows: 4,249
Resolved labeled rows: 3,195
Unique resolved host stars: 2,654
Class counts (0=false positive, 1=confirmed):
0    1229
1    1966
Name: count, dtype: int64
Features: ['koi_period', 'koi_time0bk', 'koi_impact', 'koi_duration', 'koi_depth', 'koi_prad', 'koi_teq', 'koi_insol', 'koi_steff', 'koi_slogg', 'koi_srad', 'koi_kepmag']


,validation,ROC_AUC,average_precision,balanced_accuracy,Brier_score_lower_is_better,precision,recall,F1,threshold
0,Random row-level CV,0.92619,0.94453,0.84712,0.10364,0.86715,0.91963,0.89262,0.5
1,Host-star grouped CV,0.92356,0.94376,0.84504,0.10569,0.86757,0.91302,0.88971,0.5


Saved: /kaggle/working/14_integrated_primary_metrics.csv


,fold,n_test_rows,n_test_host_stars,n_confirmed,n_false_positive,ROC_AUC,average_precision,balanced_accuracy
0,1,628,531,383,245,0.9193,0.9395,0.8534
1,2,650,531,387,263,0.9225,0.9355,0.8409
2,3,660,531,418,242,0.9260,0.9554,0.8371
3,4,628,528,400,228,0.9271,0.9496,0.8535
4,5,629,533,378,251,0.9270,0.9417,0.8422


Saved: /kaggle/working/14_grouped_fold_diagnostics.csv


In [3]:
# Paired cluster bootstrap: sample host stars with replacement, retaining all rows
# for each sampled host. This preserves within-host row dependence.
N_BOOTSTRAP = 1000
BOOTSTRAP_SEED = 2026
rng = np.random.default_rng(BOOTSTRAP_SEED)
unique_hosts = np.unique(groups)
host_to_indices = {
    host: np.flatnonzero(groups == host) for host in unique_hosts
}

metric_functions = {
    "ROC_AUC": lambda yy, pp: roc_auc_score(yy, pp),
    "average_precision": lambda yy, pp: average_precision_score(yy, pp),
    "balanced_accuracy": lambda yy, pp: balanced_accuracy_score(yy, pp >= THRESHOLD),
    "Brier_score_lower_is_better": lambda yy, pp: brier_score_loss(yy, pp)
}
boot_rows = []
for b in range(N_BOOTSTRAP):
    sampled_hosts = rng.choice(unique_hosts, size=len(unique_hosts), replace=True)
    idx = np.concatenate([host_to_indices[h] for h in sampled_hosts])
    yy = y[idx]
    if np.unique(yy).size != 2:
        continue
    for metric_name, fn in metric_functions.items():
        r_value = fn(yy, random_p[idx])
        g_value = fn(yy, grouped_p[idx])
        boot_rows.append({
            "bootstrap": b,
            "metric": metric_name,
            "random_row_cv": r_value,
            "host_grouped_cv": g_value,
            "grouped_minus_random": g_value - r_value
        })

bootstrap = pd.DataFrame(boot_rows)
summary_rows = []
for metric_name, part in bootstrap.groupby("metric"):
    delta = part["grouped_minus_random"].to_numpy()
    summary_rows.append({
        "metric": metric_name,
        "valid_bootstrap_replicates": len(delta),
        "grouped_minus_random_median": np.median(delta),
        "difference_CI_2.5_percentile": np.quantile(delta, 0.025),
        "difference_CI_97.5_percentile": np.quantile(delta, 0.975),
        "interpretation": "Cluster-bootstrap interval; descriptive uncertainty, not a causal test"
    })
bootstrap_summary = pd.DataFrame(summary_rows)
display(bootstrap_summary.round(5))
save_csv(bootstrap, "14_host_cluster_bootstrap_replicates.csv")
save_csv(bootstrap_summary, "14_host_cluster_bootstrap_summary.csv")

,metric,valid_bootstrap_replicates,grouped_minus_random_median,difference_CI_2.5_percentile,difference_CI_97.5_percentile,interpretation
0,Brier_score_lower_is_better,1000,0.00203,0.00035,0.00361,Cluster-bootstrap interval; descriptive uncert...
1,ROC_AUC,1000,-0.00261,-0.00525,0.00022,Cluster-bootstrap interval; descriptive uncert...
2,average_precision,1000,-0.00079,-0.00425,0.00310,Cluster-bootstrap interval; descriptive uncert...
3,balanced_accuracy,1000,-0.00204,-0.00950,0.00649,Cluster-bootstrap interval; descriptive uncert...


Saved: /kaggle/working/14_host_cluster_bootstrap_replicates.csv
Saved: /kaggle/working/14_host_cluster_bootstrap_summary.csv


In [4]:
# Export paired OOF predictions to support figures and reproducibility.
oof_export = pd.DataFrame({
    "row_index": labeled.index,
    "host_id": groups,
    "true_label": y,
    "random_oof_probability": random_p,
    "random_oof_fold": random_fold,
    "grouped_oof_probability": grouped_p,
    "grouped_oof_fold": grouped_fold
})
save_csv(oof_export, "14_integrated_oof_predictions.csv")

print("\nNotebook 14 complete.")
print("Primary interpretation: compare the observed metric differences and their")
print("host-cluster bootstrap intervals. Do not claim a large effect unless the")
print("actual results support that conclusion.")

Saved: /kaggle/working/14_integrated_oof_predictions.csv

Notebook 14 complete.
Primary interpretation: compare the observed metric differences and their
host-cluster bootstrap intervals. Do not claim a large effect unless the
actual results support that conclusion.


## Scientific limitations
- Random-row and grouped OOF predictions are paired on the same catalog rows, but they arise from different training/test designs.
- The bootstrap resamples host stars, not individual rows, to respect within-host clustering. It is still conditional on this catalog, feature set, model, and split seed.
- An interval containing zero does not prove the designs are equivalent; an interval excluding zero does not prove the cause is astrophysical.
- These are classification-label validation results, not independent confirmation of planets and not a completeness-corrected occurrence-rate estimate.